# Triagegeist — The Waiting-Room Blind Spot
### What a synthetic triage benchmark can and cannot tell us, and a second-read watch validated on real ED data

**Summary.**
1. The synthetic labels were generated, not assigned by people: the 50 nurses show no rater effect (Cramér's V 0.02 vs 0.35 across real hospitals), and the vitals encode the label (|r| 0.46–0.65 vs ≤ 0.08 in real triage).
2. With the same 10 vitals and the same model, acuity agreement is κ 0.90 on the synthetic data and **0.24 on real NHAMCS triage**.
3. On the synthetic data, intake fields add nothing beyond the acuity level for predicting escalation (ΔAUC +0.001), so the benchmark cannot evaluate a waiting-room tool.
4. On **real** NHAMCS data, among patients triaged as lower acuity, intake data predicts admission far better than the triage level alone: **AUC 0.80 vs 0.64** (hospital-grouped validation). Flagging the top 20% catches **58% [55, 61]** of later admissions at 2.9× the base rate, with positive net benefit for thresholds from 2% to 35%.

All intervals are 95% bootstrap (hospital-cluster on NHAMCS). Full write-up and code: linked repository.

## 1. Setup
The cells below write the project's four modules into the working directory. They are the same files as `src/` in the repository.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data', exist_ok=True)

In [ ]:
%%writefile src/stats.py
"""
Statistical helpers shared by the analysis: bootstrap confidence intervals,
cross-fitted calibration, calibration diagnostics, operating points and
decision-curve analysis.

Every interval in the write-up comes from these functions. Two bootstrap
flavours are used:
  * patient-level bootstrap (synthetic data, where visits are independent);
  * cluster bootstrap that resamples whole hospitals (NHAMCS, where visits are
    nested in hospitals and treating them as independent understates variance).
"""
from __future__ import annotations
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from sklearn.model_selection import KFold

SEED = 42


# --------------------------------------------------------------------------
# bootstrap
# --------------------------------------------------------------------------
def _resample_index(n, clusters, rng):
    """Row indices for one bootstrap replicate (cluster-aware if clusters given)."""
    if clusters is None:
        return rng.integers(0, n, n)
    uniq, inv = np.unique(clusters, return_inverse=True)
    rows_by_cluster = np.split(np.argsort(inv, kind="stable"),
                               np.cumsum(np.bincount(inv))[:-1])
    pick = rng.integers(0, len(uniq), len(uniq))
    return np.concatenate([rows_by_cluster[k] for k in pick])


def bootstrap(stat, *arrays, clusters=None, n_boot=1000, seed=SEED, alpha=0.05):
    """Point estimate + percentile CI of stat(*arrays).

    stat receives the resampled arrays and returns a float (or NaN when the
    replicate is degenerate, e.g. a single class)."""
    arrays = [np.asarray(a) for a in arrays]
    point = float(stat(*arrays))
    rng = np.random.default_rng(seed)
    n = len(arrays[0])
    reps = []
    for _ in range(n_boot):
        idx = _resample_index(n, clusters, rng)
        try:
            v = stat(*[a[idx] for a in arrays])
        except ValueError:
            v = np.nan
        reps.append(v)
    reps = np.asarray(reps, dtype=float)
    reps = reps[np.isfinite(reps)]
    lo, hi = np.quantile(reps, [alpha / 2, 1 - alpha / 2])
    return {"est": round(point, 4), "lo": round(float(lo), 4), "hi": round(float(hi), 4)}


def auc(y, p, w=None):
    return roc_auc_score(y, p, sample_weight=w)


def qwk(y, p):
    return cohen_kappa_score(y, p, weights="quadratic")


def delta_auc(y, p_new, p_base, w=None):
    return auc(y, p_new, w) - auc(y, p_base, w)


# --------------------------------------------------------------------------
# calibration
# --------------------------------------------------------------------------
def crossfit_isotonic(p, y, n_splits=5, seed=SEED):
    """Calibrate out-of-fold scores without reusing the same rows.

    The isotonic map applied to each fold is fitted only on the other folds,
    so the reported Brier score is an honest out-of-sample estimate."""
    p = np.asarray(p); y = np.asarray(y)
    out = np.zeros_like(p, dtype=float)
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(p):
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p[tr], y[tr])
        out[va] = iso.predict(p[va])
    return out


def calibration_stats(y, p, n_bins=10):
    """Brier, expected calibration error, calibration slope and
    calibration-in-the-large (mean predicted minus observed)."""
    y = np.asarray(y); p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    brier = float(np.mean((p - y) ** 2))
    edges = np.quantile(p, np.linspace(0, 1, n_bins + 1))
    b = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, n_bins - 1)
    ece = sum(abs(p[b == k].mean() - y[b == k].mean()) * (b == k).mean()
              for k in range(n_bins) if (b == k).any())
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    slope = float(LogisticRegression(C=1e6).fit(logit, y).coef_[0, 0])
    return {"brier": round(brier, 4), "ece": round(float(ece), 4),
            "slope": round(slope, 3), "citl": round(float(p.mean() - y.mean()), 4)}


# --------------------------------------------------------------------------
# operating points and decision curves
# --------------------------------------------------------------------------
def flag_top(p, frac):
    """Boolean flag for the top `frac` of scores."""
    return p >= np.quantile(p, 1 - frac)


def operating_point(y, p, frac, w=None):
    """Alert rate, precision, recall and lift when flagging the top `frac`."""
    y = np.asarray(y); p = np.asarray(p)
    w = np.ones_like(p, dtype=float) if w is None else np.asarray(w, dtype=float)
    f = flag_top(p, frac)
    prec = np.sum(w[f] * y[f]) / np.sum(w[f])
    rec = np.sum(w[f] * y[f]) / np.sum(w * y)
    base = np.sum(w * y) / np.sum(w)
    return {"alert_rate": round(float(np.sum(w[f]) / np.sum(w)), 3), "precision": round(float(prec), 3),
            "recall": round(float(rec), 3), "lift": round(float(prec / base), 2), "base_rate": round(float(base), 3)}


def recall_at(y, p, frac):
    f = flag_top(np.asarray(p), frac)
    y = np.asarray(y)
    return y[f].sum() / y.sum()


def lift_at(y, p, frac):
    f = flag_top(np.asarray(p), frac)
    y = np.asarray(y)
    return y[f].mean() / y.mean()


def net_benefit(y, flag, pt):
    """Vickers & Elkin net benefit: TP/N - FP/N * pt/(1-pt)."""
    y = np.asarray(y); flag = np.asarray(flag, dtype=bool); n = len(y)
    tp = np.sum(flag & (y == 1)); fp = np.sum(flag & (y == 0))
    return tp / n - fp / n * pt / (1 - pt)


def decision_curve(y, p, thresholds, clusters=None, n_boot=300, seed=SEED):
    """Net benefit of the model, of treating everyone and of treating no-one,
    with a bootstrap band for the model's advantage over the better default."""
    y = np.asarray(y); p = np.asarray(p)
    model = [net_benefit(y, p >= t, t) for t in thresholds]
    everyone = [net_benefit(y, np.ones_like(y, dtype=bool), t) for t in thresholds]
    rng = np.random.default_rng(seed)
    adv = []
    for _ in range(n_boot):
        idx = _resample_index(len(y), clusters, rng)
        yy, pp = y[idx], p[idx]
        adv.append([net_benefit(yy, pp >= t, t) - max(0.0, net_benefit(yy, np.ones_like(yy, dtype=bool), t))
                    for t in thresholds])
    adv = np.asarray(adv)
    return {"thresholds": [round(float(t), 3) for t in thresholds],
            "model": [round(float(v), 5) for v in model],
            "treat_all": [round(float(v), 5) for v in everyone],
            "adv_lo": [round(float(v), 5) for v in np.quantile(adv, 0.025, axis=0)],
            "adv_hi": [round(float(v), 5) for v in np.quantile(adv, 0.975, axis=0)]}


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (same order as input)."""
    p = np.asarray(pvals, dtype=float); m = len(p)
    order = np.argsort(p); adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p[i]))
        adj[i] = running
    return adj

In [ ]:
%%writefile src/load_nhamcs.py
"""
Load the NHAMCS Emergency Department public-use files (2021-2022) into one
tidy, modelling-ready table.

The raw files are fixed-width ASCII with no delimiters. Most column positions
come from data/layout<YEAR>.json (parsed from the codebook) and are copied into
FRONT/BACK below; the loader asserts they still match when the JSON is present.
The visit
disposition check-boxes are missing from those layouts, so their positions are
written out below and were checked against the codebook (doc21-ed-508.pdf p.52:
items 224-239 at bytes 485-500; doc22-ed-508.pdf p.52: items 226-241 at bytes
487-502). Two COVID items added in 2022 push them 2 bytes to the right.

Predictors are limited to what is known at triage. Disposition, diagnoses and
length of stay are kept only as outcomes. Race/ethnicity and payer are kept
only for the equity audit and never enter a model. Survey design variables
(PATWT weight, hospital code, PSU and stratum) are kept for weighted and
hospital-clustered evaluation.

Run:  python src/load_nhamcs.py   ->  data/nhamcs_ed_clean.parquet
Raw files: https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHAMCS/
           (ed2021.zip, ED2022.zip; unzip into data/ as ed2021, ed2022)
"""
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(__file__).resolve().parent.parent / "data"

VITALS = ["TEMPF", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT", "PAINSCALE"]
SENSITIVE = ["RACERETH", "PAYTYPER"]
TARGET = "IMMEDR"

# Visit disposition check-boxes (0 = no, 1 = yes), first byte of the block.
DISPOSITION = ["NODISP", "NOFU", "RETRNED", "RETREFFU", "LWBS", "LBTC", "LEFTAMA",
               "DOA", "DIEDED", "TRANNH", "TRANPSYC", "TRANOTH", "ADMITHOS",
               "OBSHOS", "OBSDIS", "OTHDISP"]
DISPOSITION_START = {2021: 485, 2022: 487}

SPECIAL_MISSING = [-9, -8, -7]
ZERO_IS_MISSING = ["TEMPF", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT"]

LABELS = {
    "SEX": {1: "Female", 2: "Male"},
    "RACERETH": {1: "NH White", 2: "NH Black", 3: "Hispanic", 4: "NH Other"},
    "PAYTYPER": {1: "Private", 2: "Medicare", 3: "Medicaid/CHIP", 4: "Workers comp",
                 5: "Self-pay", 6: "No charge/charity", 7: "Other"},
}


# Byte positions (1-based, inclusive) copied from data/layout<YEAR>.json so the
# loader also runs where the layout files are absent (e.g. a Kaggle notebook).
# Front-of-record fields are identical in 2021 and 2022.
FRONT = {"TEMPF": (48, 51), "PULSE": (52, 54), "RESPR": (55, 57), "BPSYS": (58, 60),
         "BPDIAS": (61, 63), "POPCT": (64, 66), "PAINSCALE": (69, 70), "AGE": (16, 18),
         "SEX": (25, 25), "RESIDNCE": (23, 24), "ARREMS": (33, 34), "SEEN72": (71, 72),
         "RFV1": (73, 77), "RACERETH": (32, 32), "PAYTYPER": (46, 47), "IMMEDR": (67, 68)}
BACK = {2021: {"HOSPCODE": (542, 544), "CSTRATM": (2343, 2350), "CPSUM": (2351, 2356), "PATWT": (2357, 2367)},
        2022: {"HOSPCODE": (544, 546), "CSTRATM": (2345, 2352), "CPSUM": (2353, 2358), "PATWT": (2359, 2369)}}
RAW = {2021: "ed2021.zip", 2022: "ED2022.zip"}
CDC = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHAMCS"


def _layout(year: int) -> dict:
    spec = {**FRONT, **BACK[year]}
    lay_file = DATA_DIR / f"layout{year}.json"
    if lay_file.exists():  # cross-check against the parsed codebook when available
        lay = json.loads(lay_file.read_text())
        bad = [c for c, (s, e) in spec.items() if (lay[c]["start"], lay[c]["end"]) != (s, e)]
        assert not bad, f"layout mismatch {year}: {bad}"
    start = DISPOSITION_START[year]
    for i, c in enumerate(DISPOSITION):
        spec[c] = (start + i, start + i)
    return spec


def _raw_bytes(year: int) -> bytes:
    """Raw fixed-width file: local data/ed<YEAR> if present, else download from CDC."""
    local = DATA_DIR / f"ed{year}"
    if local.exists():
        return local.read_bytes()
    import io, urllib.request, zipfile
    blob = urllib.request.urlopen(f"{CDC}/{RAW[year]}", timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(blob))
    return z.read(z.namelist()[0])


def load_year(year: int) -> pd.DataFrame:
    import io
    spec = _layout(year)
    colspecs = [(s - 1, e) for s, e in spec.values()]
    df = pd.read_fwf(io.BytesIO(_raw_bytes(year)), colspecs=colspecs, names=list(spec), dtype=str)
    df = df.apply(pd.to_numeric, errors="coerce")
    df["YEAR"] = year
    # Hospital codes are only unique within a survey year.
    df["HOSP_ID"] = df["YEAR"].astype(str) + "-" + df["HOSPCODE"].astype("Int64").astype(str)
    return df


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    n_raw = len(df)
    df["immedr_raw"] = df[TARGET]
    df["has_triage_level"] = df[TARGET].isin([1, 2, 3, 4, 5]).astype(int)

    for c in VITALS + ["SEX", "RESIDNCE", "ARREMS", "SEEN72", "AGE"] + SENSITIVE:
        df.loc[df[c].isin(SPECIAL_MISSING), c] = np.nan
    for c in ZERO_IS_MISSING:
        df.loc[df[c] == 0, c] = np.nan
    df.loc[df["PULSE"] == 998, "PULSE"] = np.nan      # Doppler / palpated
    df.loc[df["BPDIAS"] == 998, "BPDIAS"] = np.nan    # "P" or "Palp"
    df["TEMPF"] = df["TEMPF"] / 10.0                  # implied decimal: 0986 -> 98.6
    df.loc[df["PAINSCALE"] > 10, "PAINSCALE"] = np.nan
    df.loc[df["RFV1"] < 0, "RFV1"] = np.nan
    df["RFV1_module"] = (df["RFV1"] // 10000).astype("Int64")
    df["TEMPC"] = (df["TEMPF"] - 32) * 5 / 9
    df["ambulance"] = (df["ARREMS"] == 1).astype(float).where(df["ARREMS"].notna())

    # Outcomes. A visit with every disposition box blank/0 has unknown outcome.
    disp = df[DISPOSITION].fillna(0)
    df["disp_known"] = (disp.drop(columns="NODISP").sum(axis=1) > 0).astype(int)
    df["admit"] = ((disp["ADMITHOS"] == 1) | (disp["OBSHOS"] == 1)).astype(int)
    df["escalate"] = ((df["admit"] == 1) | (disp["TRANPSYC"] == 1) | (disp["TRANOTH"] == 1)
                      | (disp["DIEDED"] == 1)).astype(int)
    df["left_early"] = ((disp["LWBS"] == 1) | (disp["LBTC"] == 1)).astype(int)

    for c, mp in LABELS.items():
        df[c + "_lbl"] = df[c].map(mp)
    df.attrs["n_raw"] = n_raw
    return df.reset_index(drop=True)


def build(years=(2021, 2022), save=True) -> pd.DataFrame:
    df = clean(pd.concat([load_year(y) for y in years], ignore_index=True))
    if not save:
        return df
    out = DATA_DIR / "nhamcs_ed_clean.parquet"
    DATA_DIR.mkdir(exist_ok=True)
    df.to_parquet(out, index=False)
    print(f"saved {out}  shape={df.shape}")
    return df


if __name__ == "__main__":
    d = build()
    print("raw IMMEDR codes:", d["immedr_raw"].value_counts().sort_index().to_dict())
    print("admission rate (disposition known):", round(d.loc[d.disp_known == 1, "admit"].mean(), 3))
    print("hospitals:", d["HOSP_ID"].nunique())

In [ ]:
%%writefile src/analysis.py
"""
Triagegeist analysis pipeline. Single source of truth for every number in the
write-up: run it and it writes artifacts/results.json (plus OOF predictions in
artifacts/oof.parquet for the figures).

  M1  How the synthetic data was generated (leakage and shortcut forensics)
  M2  Acuity model: leak-free, ablated, and compared with real data on a
      harmonised feature set
  M3  Equity audit, adjusted for the true acuity mix
  M4  Waiting-room escalation watch on the synthetic queue
  M5  External validation on NHAMCS: hospital-grouped CV, survey weights,
      incremental value over the real triage level, decision curve, equity

Usage:
  python src/load_nhamcs.py      # once, builds data/nhamcs_ed_clean.parquet
  python src/analysis.py         # ~10 minutes on a laptop CPU
"""
from __future__ import annotations
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import chi2_contingency
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, GroupKFold, StratifiedGroupKFold

sys.path.insert(0, str(Path(__file__).resolve().parent))
import stats as S  # noqa: E402

ROOT = Path(__file__).resolve().parent.parent
ART = ROOT / "artifacts"
SEED = 42
N_BOOT = 1000

# ---------------------------------------------------------------- features
CAT = ["arrival_mode", "arrival_season", "shift", "sex", "language", "insurance_type",
       "transport_origin", "pain_location", "mental_status_triage",
       "chief_complaint_system", "age_group", "site_id"]
NUM = ["arrival_hour", "arrival_month", "age", "num_prior_ed_visits_12m",
       "num_prior_admissions_12m", "num_active_medications", "num_comorbidities",
       "systolic_bp", "diastolic_bp", "mean_arterial_pressure", "pulse_pressure",
       "heart_rate", "respiratory_rate", "temperature_c", "spo2", "gcs_total",
       "pain_score", "weight_kg", "height_cm", "bmi", "shock_index", "news2_score"]
VITALS = ["systolic_bp", "diastolic_bp", "mean_arterial_pressure", "pulse_pressure",
          "heart_rate", "respiratory_rate", "temperature_c", "spo2", "gcs_total",
          "shock_index", "news2_score", "mental_status_triage"]
ESCALATE = ["admitted", "transferred", "observation", "deceased"]

# Same ten variables, same units, in both datasets.
HARMONISED = ["age", "male", "temp_c", "hr", "rr", "sbp", "dbp", "spo2", "pain", "ambulance"]
NH_INTAKE = ["AGE", "SEX", "TEMPF", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT",
             "PAINSCALE", "ARREMS", "SEEN72", "RFV1_module"]


def find(name):
    for root in [ROOT / "comp_data", Path("/kaggle/input")]:
        if root.exists():
            hits = sorted(root.rglob(name), key=lambda p: len(str(p)))
            if hits:
                return hits[0]
    raise FileNotFoundError(name)


def load_synthetic():
    tr = pd.read_csv(find("train.csv"))
    hist = pd.read_csv(find("patient_history.csv"))
    cc = pd.read_csv(find("chief_complaints.csv"))
    hx = [c for c in hist.columns if c.startswith("hx_")]
    df = (tr.merge(hist[["patient_id"] + hx], on="patient_id", how="left")
            .merge(cc[["patient_id", "chief_complaint_raw"]], on="patient_id", how="left"))
    df["pain_score"] = df["pain_score"].replace(-1, np.nan)
    df["hx_burden"] = df[hx].sum(axis=1)
    df["escalate"] = df["disposition"].isin(ESCALATE).astype(int)
    df["lwbs"] = (df["disposition"] == "lwbs").astype(int)
    df["complaint_base"] = df["chief_complaint_raw"].str.split(r",|—|–| with ", regex=True).str[0].str.strip()
    return df, hx


def make_X(df, hx, cols=None, with_acuity=False):
    num = [c for c in (cols or NUM + hx + ["hx_burden"]) if c not in CAT]
    cat = [c for c in (cols or CAT) if c in CAT]
    X = df[num].copy()
    for c in cat:
        X[c] = df[c].astype("category")
    if with_acuity:
        X["triage_acuity"] = df["triage_acuity"]
    return X


def harmonise_synthetic(df):
    return pd.DataFrame({
        "age": df["age"], "male": (df["sex"] == "M").astype(float),
        "temp_c": df["temperature_c"], "hr": df["heart_rate"], "rr": df["respiratory_rate"],
        "sbp": df["systolic_bp"], "dbp": df["diastolic_bp"], "spo2": df["spo2"],
        "pain": df["pain_score"], "ambulance": (df["arrival_mode"] == "ambulance").astype(float)})


def harmonise_nhamcs(nd):
    return pd.DataFrame({
        "age": nd["AGE"], "male": (nd["SEX"] == 2).astype(float).where(nd["SEX"].notna()),
        "temp_c": nd["TEMPC"], "hr": nd["PULSE"], "rr": nd["RESPR"], "sbp": nd["BPSYS"],
        "dbp": nd["BPDIAS"], "spo2": nd["POPCT"], "pain": nd["PAINSCALE"], "ambulance": nd["ambulance"]})


# ---------------------------------------------------------------- models
def clf(**kw):
    p = dict(n_estimators=300, learning_rate=0.05, num_leaves=48, subsample=0.8, subsample_freq=1,
             colsample_bytree=0.8, min_child_samples=40, random_state=SEED, verbose=-1)
    p.update(kw)
    return lgb.LGBMClassifier(**p)


def oof_multiclass(X, y, splitter, groups=None):
    oof = np.zeros(len(y), dtype=int)
    for tr, va in splitter.split(X, y, groups):
        oof[va] = clf().fit(X.iloc[tr], y[tr]).predict(X.iloc[va])
    return oof


def oof_binary(X, y, splitter, groups=None):
    """Out-of-fold probabilities. No class re-weighting, so the raw scores are
    already on a probability scale; ranking metrics are unaffected either way."""
    oof = np.zeros(len(y))
    for tr, va in splitter.split(X, y, groups):
        m = clf(n_estimators=400, learning_rate=0.03).fit(X.iloc[tr], y[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return oof


def acuity_metrics(y, pred, clusters=None, n_boot=N_BOOT):
    high = y <= 2
    return {"accuracy": round(float(accuracy_score(y, pred)), 4),
            "qwk": S.bootstrap(S.qwk, y, pred, clusters=clusters, n_boot=n_boot),
            "undertriage_rate": round(float((pred > y).mean()), 4),
            "overtriage_rate": round(float((pred < y).mean()), 4),
            "high_acuity_recall": round(float((pred[high] <= 2).mean()), 4),
            "n_missed_high_acuity": int(((pred > 2) & high).sum())}


def cramers_v(table):
    chi2, p, _, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    k = min(table.shape) - 1
    return float(np.sqrt(chi2 / (n * k))), float(p)


def news2_from_vitals(df):
    def band(x, cuts, pts):
        return pd.Series(np.select([x <= c for c in cuts], pts[:-1], pts[-1]), index=x.index).where(x.notna()).fillna(0)
    return (band(df["respiratory_rate"], [8, 11, 20, 24], [3, 1, 0, 2, 3])
            + band(df["spo2"], [91, 93, 95], [3, 2, 1, 0])
            + band(df["systolic_bp"], [90, 100, 110, 219], [3, 2, 1, 0, 3])
            + band(df["heart_rate"], [40, 50, 90, 110, 130], [3, 1, 0, 1, 2, 3])
            + band(df["temperature_c"], [35.0, 36.0, 38.0, 39.0], [3, 1, 0, 1, 2])
            + 3 * (df["mental_status_triage"] != "alert").astype(int))


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ===================================================================== M1
def m1_forensics(df, nd):
    log("M1 forensics")
    r = {}
    r["corr_with_acuity"] = {c: round(float(df[c].corr(df["triage_acuity"])), 3)
                             for c in ["ed_los_hours", "news2_score", "shock_index", "gcs_total", "spo2", "heart_rate"]}
    # Real-data reference: same vitals vs real triage level
    t = nd[nd.has_triage_level == 1]
    r["corr_with_acuity_nhamcs"] = {c: round(float(t[c].corr(t["IMMEDR"])), 3)
                                    for c in ["PULSE", "RESPR", "POPCT", "BPSYS", "TEMPF"]}
    r["corr_with_acuity_synthetic_same_vitals"] = {
        k: round(float(df[v].corr(df["triage_acuity"])), 3)
        for k, v in {"PULSE": "heart_rate", "RESPR": "respiratory_rate", "POPCT": "spo2",
                     "BPSYS": "systolic_bp", "TEMPF": "temperature_c"}.items()}

    # Complaint text -> acuity purity, overall and for frequent complaints only
    g = df.groupby("chief_complaint_raw")["triage_acuity"]
    cnt = g.size()
    pur = g.agg(lambda s: s.value_counts(normalize=True).iloc[0])
    r["complaints"] = {"n_unique": int(cnt.size), "median_count": float(cnt.median()),
                       "purity_all": round(float(pur.mean()), 4),
                       "share_deterministic_all": round(float((pur == 1).mean()), 4),
                       "n_with_20plus": int((cnt >= 20).sum()),
                       "purity_20plus": round(float(pur[cnt >= 20].mean()), 4),
                       "share_deterministic_20plus": round(float((pur[cnt >= 20] == 1).mean()), 4)}

    # NEWS2 recomputed from the vitals with the RCP 2017 bands
    calc = news2_from_vitals(df)
    r["news2_recomputed"] = {"exact_match": round(float((calc == df["news2_score"]).mean()), 3),
                             "corr_given_vs_recomputed": round(float(calc.corr(df["news2_score"])), 3),
                             "corr_recomputed_with_acuity": round(float(calc.corr(df["triage_acuity"])), 3)}

    # Rater effect. Real triage varies by nurse and by hospital; generated labels do not.
    nurse_tab = pd.crosstab(df["triage_nurse_id"], df["triage_acuity"])
    v_n, p_n = cramers_v(nurse_tab)
    hosp = t[t.groupby("HOSP_ID")["IMMEDR"].transform("size") >= 20]
    v_h, p_h = cramers_v(pd.crosstab(hosp["HOSP_ID"], hosp["IMMEDR"]))
    r["rater_effect"] = {"synthetic_nurses": int(nurse_tab.shape[0]),
                         "synthetic_nurse_cramers_v": round(v_n, 4), "synthetic_nurse_p": round(p_n, 4),
                         "nhamcs_hospitals": int(hosp["HOSP_ID"].nunique()),
                         "nhamcs_hospital_cramers_v": round(v_h, 4), "nhamcs_hospital_p": float(f"{p_h:.2e}")}

    # Text shortcut: a bag-of-words model with and without template hold-out
    text = df["chief_complaint_raw"].fillna("")
    y = df["triage_acuity"].values
    Xt = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(text)
    def text_oof(splitter, groups=None):
        oof = np.zeros(len(y), dtype=int)
        for tr, va in splitter.split(Xt, y, groups):
            m = LogisticRegression(C=5, max_iter=2000).fit(Xt[tr], y[tr])
            oof[va] = m.predict(Xt[va])
        return oof
    r["text_model"] = {}
    for name, sp, grp in [("stratified", StratifiedKFold(5, shuffle=True, random_state=SEED), None),
                          ("group_exact_complaint", GroupKFold(5), df["chief_complaint_raw"].values),
                          ("group_complaint_base", GroupKFold(5), df["complaint_base"].values)]:
        o = text_oof(sp, grp)
        r["text_model"][name] = {"accuracy": round(float(accuracy_score(y, o)), 4), "qwk": round(float(S.qwk(y, o)), 4)}
    r["text_model"]["n_complaint_bases"] = int(df["complaint_base"].nunique())
    return r


# ===================================================================== M2
def m2_acuity(df, hx, nd):
    log("M2 acuity models")
    y = df["triage_acuity"].values
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    r = {}
    X = make_X(df, hx)
    oof_skf = oof_multiclass(X, y, skf)
    r["structured_stratified"] = acuity_metrics(y, oof_skf, n_boot=300)
    oof_gkf = oof_multiclass(X, y, GroupKFold(5), df["complaint_base"].values)
    r["structured_group_by_complaint_base"] = acuity_metrics(y, oof_gkf, n_boot=300)
    oof_site = oof_multiclass(X, y, GroupKFold(5), df["site_id"].values)
    r["structured_group_by_site"] = acuity_metrics(y, oof_site, n_boot=300)

    # Ablations: what carries the label?
    vit_only = make_X(df, hx, cols=VITALS)
    r["ablation_vitals_only"] = acuity_metrics(y, oof_multiclass(vit_only, y, skf), n_boot=300)
    no_vit = make_X(df, hx, cols=[c for c in NUM + CAT + hx + ["hx_burden"] if c not in VITALS])
    r["ablation_without_vitals"] = acuity_metrics(y, oof_multiclass(no_vit, y, skf), n_boot=300)

    # Harmonised comparison: identical 10 features, identical model, both datasets
    Hs = harmonise_synthetic(df)
    oof_hs = oof_multiclass(Hs, y, skf)
    r["harmonised_synthetic"] = acuity_metrics(y, oof_hs, n_boot=300)
    t = nd[nd.has_triage_level == 1].reset_index(drop=True)
    Hn = harmonise_nhamcs(t); yn = t["IMMEDR"].astype(int).values
    oof_hn = oof_multiclass(Hn, yn, StratifiedGroupKFold(5, shuffle=True, random_state=SEED), t["HOSP_ID"].values)
    r["harmonised_nhamcs"] = acuity_metrics(yn, oof_hn, clusters=t["HOSP_ID"].values)
    # Original 12-feature NHAMCS set, hospital-grouped vs random split
    Xn = t[NH_INTAKE]
    r["nhamcs_intake_random_split"] = acuity_metrics(yn, oof_multiclass(Xn, yn, skf), clusters=t["HOSP_ID"].values)
    r["nhamcs_intake_hospital_split"] = acuity_metrics(
        yn, oof_multiclass(Xn, yn, StratifiedGroupKFold(5, shuffle=True, random_state=SEED), t["HOSP_ID"].values),
        clusters=t["HOSP_ID"].values)
    return r, oof_skf


# ===================================================================== M3
def oe_by_group(under, acuity, group):
    """Observed / expected undertriage per group, where 'expected' uses the
    overall rate at each true acuity level (indirect standardisation)."""
    d = pd.DataFrame({"u": under, "a": acuity, "g": group})
    rate_at = d.groupby("a")["u"].mean()
    d["e"] = d["a"].map(rate_at)
    s = d.groupby("g").agg(n=("u", "size"), obs=("u", "sum"), exp=("e", "sum"))
    return s["obs"] / s["exp"], s


def m3_equity(df, oof_skf):
    log("M3 equity audit")
    y = df["triage_acuity"].values
    under = (oof_skf > y).astype(int)
    rng = np.random.default_rng(SEED)
    rate_at = pd.Series(under).groupby(y).mean()
    exp_row = pd.Series(y).map(rate_at).to_numpy()
    out, pvals = {}, []

    def heterogeneity(gg):
        oe, s = oe_by_group(under, y, gg)
        return float(np.average((oe - 1) ** 2, weights=s["n"]))

    for col in ["language", "insurance_type", "sex", "age_group"]:
        g = df[col].astype(str).values
        oe, s = oe_by_group(under, y, g)
        crude = pd.Series(under).groupby(g).mean()
        rows = {}
        for grp in oe.index:
            m = g == grp
            ci = S.bootstrap(lambda u, e: float(u.sum() / e.sum()), under[m], exp_row[m], n_boot=500)
            rows[grp] = {"n": int(s.loc[grp, "n"]), "crude_rate": round(float(crude[grp]), 4),
                         "oe_ratio": ci}
        # Permutation test of heterogeneity: shuffle group labels within each acuity level
        stat = heterogeneity
        obs = stat(g)
        null = []
        for _ in range(300):
            gp = g.copy()
            for a in np.unique(y):
                idx = np.where(y == a)[0]
                gp[idx] = rng.permutation(gp[idx])
            null.append(stat(gp))
        p = (1 + np.sum(np.asarray(null) >= obs)) / (1 + len(null))
        pvals.append(p)
        out[col] = {"groups": rows, "perm_p": round(float(p), 4)}
    for col, p_adj in zip(out, S.holm(pvals)):
        out[col]["holm_p"] = round(float(p_adj), 4)
    # Why crude rates mislead: undertriage is impossible at ESI 5
    out["undertriage_rate_by_true_acuity"] = {int(k): round(float(v), 4)
                                              for k, v in pd.Series(under).groupby(y).mean().items()}
    return out


# ===================================================================== M4
def m4_waiting_room(df, hx):
    log("M4 waiting-room watch (synthetic)")
    ye = df["escalate"].values
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    X_ac = df[["triage_acuity"]]
    X_full = make_X(df, hx, with_acuity=True)
    p_ac = oof_binary(X_ac, ye, skf)
    p_full = oof_binary(X_full, ye, skf)
    p_site = oof_binary(X_full, ye, GroupKFold(5), df["site_id"].values)
    low = df["triage_acuity"].isin([3, 4, 5]).values
    r = {"prevalence_all": round(float(ye.mean()), 4), "prevalence_low": round(float(ye[low].mean()), 4),
         "n_low": int(low.sum())}
    r["auc_all"] = {"acuity_only": S.bootstrap(S.auc, ye, p_ac),
                    "full": S.bootstrap(S.auc, ye, p_full),
                    "delta_full_minus_acuity": S.bootstrap(S.delta_auc, ye, p_full, p_ac)}
    yl, pl, pal = ye[low], p_full[low], p_ac[low]
    r["auc_low"] = {"acuity_only": S.bootstrap(S.auc, yl, pal),
                    "full": S.bootstrap(S.auc, yl, pl),
                    "delta_full_minus_acuity": S.bootstrap(S.delta_auc, yl, pl, pal),
                    "full_site_held_out": S.bootstrap(S.auc, yl, p_site[low])}
    r["operating_points_low"] = {}
    for frac in [0.05, 0.10, 0.20]:
        op = S.operating_point(yl, pl, frac)
        op["recall_ci"] = S.bootstrap(lambda a, b: S.recall_at(a, b, frac), yl, pl, n_boot=500)
        op["lift_ci"] = S.bootstrap(lambda a, b: S.lift_at(a, b, frac), yl, pl, n_boot=500)
        r["operating_points_low"][f"top_{int(frac * 100)}pct"] = op
    cal = S.crossfit_isotonic(p_full, ye)
    r["calibration"] = {"raw": S.calibration_stats(ye, p_full), "crossfit_isotonic": S.calibration_stats(ye, cal)}

    yw = df["lwbs"].values
    p_l = oof_binary(X_full, yw, skf)
    r["lwbs"] = {"prevalence": round(float(yw.mean()), 4), "auc": S.bootstrap(S.auc, yw, p_l, n_boot=500)}

    # Demo cases, chosen by rule and labelled as such
    thr20 = np.quantile(cal[low], 0.80)
    flagged_tp = np.where(low & (cal >= thr20) & (ye == 1))[0]
    flagged = np.where(low & (cal >= thr20))[0]
    typical = flagged_tp[np.argsort(cal[flagged_tp])[len(flagged_tp) // 2]]
    best = flagged_tp[np.argmax(cal[flagged_tp])]
    fp_typ = np.setdiff1d(flagged, flagged_tp)
    fp_typical = fp_typ[np.argsort(cal[fp_typ])[len(fp_typ) // 2]]
    m = clf(n_estimators=400, learning_rate=0.03).fit(X_full, ye)
    def explain(i):
        c = np.asarray(m.booster_.predict(X_full.iloc[[i]], pred_contrib=True)).ravel()[:-1]
        top = sorted(zip(X_full.columns, c), key=lambda t: -abs(t[1]))[:4]
        return {"patient_id": df["patient_id"].iloc[i], "acuity": int(df["triage_acuity"].iloc[i]),
                "risk": round(float(cal[i]), 3), "escalated": int(ye[i]),
                "disposition": df["disposition"].iloc[i],
                "drivers": [(n, round(float(v), 3)) for n, v in top]}
    r["demo"] = {"threshold_top20": round(float(thr20), 3),
                 "precision_of_flags": round(float(ye[flagged].mean()), 3),
                 "typical_true_positive": explain(typical),
                 "typical_false_positive": explain(fp_typical),
                 "highest_risk_true_positive": explain(best)}
    oof = pd.DataFrame({"p_acuity": p_ac, "p_full": p_full, "p_cal": cal, "y": ye,
                        "acuity": df["triage_acuity"].values})
    return r, oof


# ===================================================================== M5
def m5_nhamcs(nd):
    log("M5 NHAMCS external validation")
    t = nd[(nd.has_triage_level == 1) & (nd.disp_known == 1)].reset_index(drop=True)
    hosp = t["HOSP_ID"].values; w = t["PATWT"].values
    sgkf = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    X_imm = t[["IMMEDR"]]
    X_int = t[NH_INTAKE]
    X_both = t[NH_INTAKE + ["IMMEDR"]]
    low = t["IMMEDR"].isin([3, 4, 5]).values
    r = {"n": int(len(t)), "n_hospitals": int(t["HOSP_ID"].nunique()), "n_low": int(low.sum()),
         "share_visits_without_triage_level": round(float(1 - nd["has_triage_level"].mean()), 3),
         "admit_rate_with_vs_without_level": [round(float(nd.loc[nd.has_triage_level == 1, "admit"].mean()), 3),
                                              round(float(nd.loc[nd.has_triage_level == 0, "admit"].mean()), 3)]}
    oofs = {}
    for target in ["admit", "escalate"]:
        y = t[target].values
        p_imm = oof_binary(X_imm, y, sgkf, hosp)
        p_int = oof_binary(X_int, y, sgkf, hosp)
        p_both = oof_binary(X_both, y, sgkf, hosp)
        p_both_rand = oof_binary(X_both, y, skf)
        yl = y[low]; wl = w[low]; hl = hosp[low]
        res = {"prevalence": round(float(y.mean()), 4), "prevalence_low": round(float(yl.mean()), 4),
               "prevalence_low_weighted": round(float(np.average(yl, weights=wl)), 4)}
        res["auc_low"] = {
            "immedr_only": S.bootstrap(S.auc, yl, p_imm[low], clusters=hl),
            "intake_only": S.bootstrap(S.auc, yl, p_int[low], clusters=hl),
            "intake_plus_immedr": S.bootstrap(S.auc, yl, p_both[low], clusters=hl),
            "delta_vs_immedr_only": S.bootstrap(S.delta_auc, yl, p_both[low], p_imm[low], clusters=hl),
            "intake_plus_immedr_weighted": S.bootstrap(S.auc, yl, p_both[low], wl, clusters=hl),
            "intake_plus_immedr_random_split": S.bootstrap(S.auc, yl, p_both_rand[low], clusters=hl)}
        res["auc_all"] = {
            "immedr_only": S.bootstrap(S.auc, y, p_imm, clusters=hosp),
            "intake_plus_immedr": S.bootstrap(S.auc, y, p_both, clusters=hosp),
            "delta_vs_immedr_only": S.bootstrap(S.delta_auc, y, p_both, p_imm, clusters=hosp)}
        pl = p_both[low]
        ops = {}
        for frac in [0.10, 0.20]:
            op = S.operating_point(yl, pl, frac)
            op["weighted"] = S.operating_point(yl, pl, frac, wl)
            op["recall_ci"] = S.bootstrap(lambda a, b: S.recall_at(a, b, frac), yl, pl, clusters=hl)
            op["lift_ci"] = S.bootstrap(lambda a, b: S.lift_at(a, b, frac), yl, pl, clusters=hl)
            ops[f"top_{int(frac * 100)}pct"] = op
        res["operating_points_low"] = ops
        cal = S.crossfit_isotonic(p_both, y)
        res["calibration"] = {"raw": S.calibration_stats(y, p_both), "crossfit_isotonic": S.calibration_stats(y, cal)}
        if target == "admit":
            ths = np.round(np.arange(0.02, 0.41, 0.01), 2)
            res["decision_curve_low"] = S.decision_curve(yl, cal[low], ths, clusters=hl)
            dc = res["decision_curve_low"]
            pos = [th for th, lo in zip(dc["thresholds"], dc["adv_lo"]) if lo > 0]
            res["decision_curve_low"]["thresholds_with_ci_above_zero"] = [min(pos), max(pos)] if pos else None
            oofs = {"y": y, "p_imm": p_imm, "p_both": p_both, "p_cal": cal, "low": low, "hosp": hosp, "w": w,
                    "race": t["RACERETH_lbl"].values, "payer": t["PAYTYPER_lbl"].values}
        r[target] = res

    # Leaving early: does the LWBS null result replicate on real data?
    yle = t["left_early"].values
    p_le = oof_binary(X_both, yle, sgkf, hosp)
    r["left_early"] = {"prevalence": round(float(yle.mean()), 4), "auc": S.bootstrap(S.auc, yle, p_le, clusters=hosp)}

    # Equity of the watch on real data: who gets caught, and is risk calibrated per group?
    eq = {}
    y, pc, lw = oofs["y"], oofs["p_cal"], oofs["low"]
    flag = np.zeros(len(y), dtype=bool)
    flag[lw] = S.flag_top(pc[lw], 0.20)
    for col, vals in [("race_ethnicity", oofs["race"]), ("payer", oofs["payer"])]:
        rows = {}
        for g in pd.Series(vals[lw]).dropna().unique():
            m = lw & (vals == g)
            if m.sum() < 200 or y[m].sum() < 20:
                continue
            rows[str(g)] = {"n": int(m.sum()), "admissions": int(y[m].sum()),
                            "recall_of_watch": S.bootstrap(lambda a, f: a[f.astype(bool)].sum() / a.sum(),
                                                           y[m], flag[m], clusters=oofs["hosp"][m], n_boot=500),
                            "observed_rate": round(float(y[m].mean()), 3),
                            "mean_predicted": round(float(pc[m].mean()), 3)}
        eq[col] = rows
    r["equity_low_acuity_watch"] = eq
    return r, oofs


# ===================================================================== main
def main():
    ART.mkdir(exist_ok=True)
    nd = pd.read_parquet(ROOT / "data" / "nhamcs_ed_clean.parquet")
    df, hx = load_synthetic()
    out = {"meta": {"seed": SEED, "n_boot": N_BOOT, "n_train": int(len(df)),
                    "lightgbm": lgb.__version__}}
    out["m1_forensics"] = m1_forensics(df, nd)
    out["m2_acuity"], oof_skf = m2_acuity(df, hx, nd)
    out["m3_equity"] = m3_equity(df, oof_skf)
    out["m4_waiting_room"], oof4 = m4_waiting_room(df, hx)
    out["m5_nhamcs"], oof5 = m5_nhamcs(nd)
    (ART / "results.json").write_text(json.dumps(out, indent=2, default=str))
    oof4.to_parquet(ART / "oof_synthetic.parquet", index=False)
    pd.DataFrame({k: v for k, v in oof5.items()}).to_parquet(ART / "oof_nhamcs.parquet", index=False)
    log("saved artifacts/results.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/figures.py
"""
Figures for the notebook and write-up, drawn only from artifacts/results.json
and the saved out-of-fold predictions, so every plotted number matches the
reported one.

  python src/figures.py   ->  artifacts/fig_*.png
"""
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parent.parent
ART = ROOT / "artifacts"
NAVY, RED, BLUE, GREEN, AMBER, GREY = "#0b1f33", "#d0453a", "#2b6cb0", "#2b8a3e", "#e08a2c", "#9aa5b1"

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 140, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})


def _err(ci):
    return [[ci["est"] - ci["lo"]], [ci["hi"] - ci["est"]]]


def fig_generator(R):
    """Two signatures of generated labels: vitals track acuity far more tightly
    than in real triage, and the label shows no rater effect."""
    f = R["m1_forensics"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    names = ["PULSE", "RESPR", "POPCT", "BPSYS", "TEMPF"]
    pretty = ["heart rate", "resp. rate", "SpO2", "systolic BP", "temperature"]
    syn = [abs(f["corr_with_acuity_synthetic_same_vitals"][n]) for n in names]
    real = [abs(f["corr_with_acuity_nhamcs"][n]) for n in names]
    x = np.arange(len(names))
    ax[0].bar(x - 0.2, syn, 0.4, color=RED, label="synthetic")
    ax[0].bar(x + 0.2, real, 0.4, color=GREEN, label="real (NHAMCS)")
    ax[0].set_xticks(x); ax[0].set_xticklabels(pretty, rotation=20)
    ax[0].set_ylabel("|correlation| with triage level"); ax[0].set_ylim(0, 0.85)
    ax[0].legend(fontsize=8, ncol=2, loc="upper center")
    ax[0].set_title("Vitals track the label far too tightly")
    r = f["rater_effect"]
    vals = [r["synthetic_nurse_cramers_v"], r["nhamcs_hospital_cramers_v"]]
    ax[1].bar([f"synthetic\n{r['synthetic_nurses']} nurses", f"real\n{r['nhamcs_hospitals']} hospitals"],
              vals, color=[RED, GREEN], width=0.5)
    for i, v in enumerate(vals):
        ax[1].text(i, v + 0.005, f"V = {v:.3f}", ha="center")
    ax[1].set_ylabel("Cramér's V (rater × triage level)")
    ax[1].set_title("No rater effect in the synthetic labels")
    fig.tight_layout()
    return fig


def fig_acuity(R):
    m = R["m2_acuity"]; t = R["m1_forensics"]["text_model"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.7))
    rows = [("synthetic\nstructured", m["structured_stratified"]["qwk"], RED),
            ("synthetic\n10 shared vitals", m["harmonised_synthetic"]["qwk"], AMBER),
            ("real NHAMCS\n10 shared vitals", m["harmonised_nhamcs"]["qwk"], GREEN)]
    for i, (lab, ci, c) in enumerate(rows):
        ax[0].bar(i, ci["est"], color=c, yerr=_err(ci), capsize=4, width=0.6)
        ax[0].text(i, ci["hi"] + 0.02, f"{ci['est']:.2f}", ha="center")
    ax[0].axhspan(0.6, 0.8, color=GREY, alpha=0.25, label="published nurse–nurse agreement")
    ax[0].set_xticks(range(3)); ax[0].set_xticklabels([r[0] for r in rows])
    ax[0].set_ylim(0, 1.05); ax[0].set_ylabel("quadratic-weighted κ (95% CI)")
    ax[0].set_title("Same model, same features: synthetic vs real"); ax[0].legend(fontsize=8, loc="upper right")
    labs = ["random folds", "unseen exact\ncomplaint", "unseen complaint\nbase phrase"]
    keys = ["stratified", "group_exact_complaint", "group_complaint_base"]
    vals = [t[k]["qwk"] for k in keys]
    ax[1].bar(labs, vals, color=[RED, AMBER, BLUE], width=0.6)
    for i, v in enumerate(vals):
        ax[1].text(i, v + 0.02, f"{v:.2f}", ha="center")
    ax[1].set_ylim(0, 1.05); ax[1].set_ylabel("κ, complaint text only")
    ax[1].set_title("The text shortcut is template memorisation")
    fig.tight_layout()
    return fig


def fig_equity(R):
    e = R["m3_equity"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, col in zip(axes, ["language", "insurance_type"]):
        g = e[col]["groups"]
        items = sorted(g.items(), key=lambda kv: kv[1]["oe_ratio"]["est"])
        for i, (name, v) in enumerate(items):
            ci = v["oe_ratio"]
            ax.errorbar(ci["est"], i, xerr=[[ci["est"] - ci["lo"]], [ci["hi"] - ci["est"]]],
                        fmt="o", color=NAVY, capsize=3)
        ax.axvline(1, color=RED, ls="--", lw=1)
        ax.set_yticks(range(len(items)))
        ax.set_yticklabels([f"{k} (n={v['n']:,})" for k, v in items])
        ax.set_xlabel("observed / expected undertriage (95% CI)")
        ax.set_title(f"{col.replace('_', ' ')} · Holm p = {e[col]['holm_p']:.2f}")
    fig.tight_layout()
    return fig


def _decile_rates(y, p, n=10):
    d = pd.qcut(pd.Series(p).rank(method="first"), n, labels=False)
    return pd.Series(y).groupby(d).mean().to_numpy()


def fig_watch(R, oof_s, oof_n):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    low_s = oof_s["acuity"].isin([3, 4, 5]).to_numpy()
    rs = _decile_rates(oof_s["y"].to_numpy()[low_s], oof_s["p_cal"].to_numpy()[low_s])
    low_n = oof_n["low"].to_numpy().astype(bool)
    rn = _decile_rates(oof_n["y"].to_numpy()[low_n], oof_n["p_cal"].to_numpy()[low_n])
    s4 = R["m4_waiting_room"]["auc_low"]; s5 = R["m5_nhamcs"]["admit"]["auc_low"]
    t_syn = (f"Synthetic ESI 3–5: steps = ESI 5/4/3\n"
             f"AUC {s4['full']['est']:.2f} vs acuity alone {s4['acuity_only']['est']:.2f}")
    t_real = (f"Real NHAMCS IMMEDR 3–5\n"
              f"AUC {s5['intake_plus_immedr']['est']:.2f} vs triage level alone {s5['immedr_only']['est']:.2f}")
    for a, r, base, title, c in [(ax[0], rs, R["m4_waiting_room"]["prevalence_low"], t_syn, BLUE),
                                 (ax[1], rn, R["m5_nhamcs"]["admit"]["prevalence_low"], t_real, GREEN)]:
        a.bar(range(1, 11), r, color=c)
        a.axhline(base, color="k", ls="--", lw=1, label=f"queue base rate {base:.1%}")
        a.set_xlabel("predicted-risk decile (1 = lowest)"); a.set_ylabel("observed escalation / admission")
        a.set_title(title); a.legend(fontsize=8); a.set_xticks(range(1, 11))
    fig.suptitle("Risk inside the 'can wait' queue: re-sorted acuity (synthetic) vs new signal (real)",
                 fontweight="bold")
    fig.tight_layout()
    return fig


def fig_decision_curve(R):
    dc = R["m5_nhamcs"]["admit"]["decision_curve_low"]
    th = np.array(dc["thresholds"])
    model = np.array(dc["model"]); allv = np.array(dc["treat_all"])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(th, model, color=GREEN, lw=2.2, label="waiting-room watch")
    ax[0].plot(th, allv, color=GREY, ls="--", label="review everyone")
    ax[0].axhline(0, color="k", lw=1, label="review no-one")
    ax[0].set_ylim(min(-0.01, model.min() * 1.1), max(model.max(), allv.max()) * 1.15)
    ax[0].set_xlabel("threshold probability"); ax[0].set_ylabel("net benefit")
    ax[0].set_title("Decision curve, real low-acuity queue"); ax[0].legend(fontsize=8)
    adv = model - np.maximum(allv, 0)
    ax[1].fill_between(th, dc["adv_lo"], dc["adv_hi"], color=GREEN, alpha=0.25, label="95% hospital-cluster CI")
    ax[1].plot(th, adv, color=GREEN, lw=2)
    ax[1].axhline(0, color="k", lw=1)
    ax[1].set_xlabel("threshold probability"); ax[1].set_ylabel("net benefit over best default")
    ax[1].set_title("Advantage over the better default policy"); ax[1].legend(fontsize=8)
    fig.tight_layout()
    return fig


def fig_real_equity(R):
    eq = R["m5_nhamcs"]["equity_low_acuity_watch"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    overall = R["m5_nhamcs"]["admit"]["operating_points_low"]["top_20pct"]["recall"]
    for ax, col in zip(axes, ["race_ethnicity", "payer"]):
        items = sorted(eq[col].items(), key=lambda kv: kv[1]["recall_of_watch"]["est"])
        for i, (name, v) in enumerate(items):
            ci = v["recall_of_watch"]
            ax.errorbar(ci["est"], i, xerr=[[ci["est"] - ci["lo"]], [ci["hi"] - ci["est"]]],
                        fmt="o", color=NAVY, capsize=3)
        ax.axvline(overall, color=GREEN, ls="--", lw=1, label=f"overall {overall:.0%}")
        ax.set_yticks(range(len(items)))
        ax.set_yticklabels([f"{k} ({v['admissions']} adm.)" for k, v in items])
        ax.set_xlabel("share of admissions caught by top-20% watch")
        ax.set_title(col.replace("_", " ")); ax.legend(fontsize=8)
    fig.suptitle("Real-data equity check of the watch", fontweight="bold")
    fig.tight_layout()
    return fig


def all_figures():
    R = json.loads((ART / "results.json").read_text())
    oof_s = pd.read_parquet(ART / "oof_synthetic.parquet")
    oof_n = pd.read_parquet(ART / "oof_nhamcs.parquet")
    return {"generator": fig_generator(R), "acuity": fig_acuity(R), "equity": fig_equity(R),
            "watch": fig_watch(R, oof_s, oof_n), "decision_curve": fig_decision_curve(R),
            "real_equity": fig_real_equity(R)}


if __name__ == "__main__":
    matplotlib.use("Agg")
    for name, fig in all_figures().items():
        fig.savefig(ART / f"fig_{name}.png", bbox_inches="tight")
        print("saved", ART / f"fig_{name}.png")

In [ ]:
import sys, json, warnings
sys.path.insert(0, "src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import stats as S, load_nhamcs, analysis as A, figures as F

nd = load_nhamcs.build()            # downloads ed2021.zip / ED2022.zip from the CDC
df, hx = A.load_synthetic()
R = {"meta": {"seed": A.SEED, "n_train": len(df)}}
print("synthetic:", df.shape, "| NHAMCS:", nd.shape)

## 2. Clinical problem
Triage assigns an acuity level (ESI 1–5) that sets how long a patient may wait. The dangerous error is undertriage. The level is a snapshot: it does not say which patients in the lower-acuity queue will need a hospital bed. We ask (1) whether this synthetic benchmark can judge triage AI, and (2) whether triage-time data identifies future admissions among lower-acuity patients **beyond the triage level itself**.

**Data.** Triagegeist synthetic ED data (80k visits, 5 sites, 50 nurses) and NHAMCS ED 2021–2022 (CDC/NCHS, public domain, 32,232 real visits). The NHAMCS fixed-width files are parsed with codebook byte positions; the disposition block moves 2 bytes between 2021 and 2022. Only triage-time information is used as a predictor.

## 3. Module 1 — How the synthetic labels were made

In [ ]:
R["m1_forensics"] = A.m1_forensics(df, nd)
m1 = R["m1_forensics"]
print(json.dumps({k: m1[k] for k in ["rater_effect", "news2_recomputed", "complaints", "text_model"]}, indent=1))
F.fig_generator(R); plt.show()

- **No rater effect.** Fifty nurses with ~1,600 patients each produce indistinguishable acuity distributions (p ≈ 0.99). Real triage differs strongly between hospitals. Labels without rater variation were not assigned by people.
- **Vitals encode the label.** Recomputing NEWS2 from the vitals reproduces the provided score (r = 0.95), so NEWS2 is not separately leaked; the vitals themselves were sampled from acuity.
- **Text is a lookup table.** 99.7% of complaints map to one level. Holding out exact strings does not help because templates share modifiers; holding out base phrases does.

## 4. Module 2 — Acuity prediction does not transfer

In [ ]:
R["m2_acuity"], oof_acuity = A.m2_acuity(df, hx, nd)
for k, v in R["m2_acuity"].items():
    q = v["qwk"]; print(f"{k:38s} acc={v['accuracy']:.3f}  kappa={q['est']:.3f} [{q['lo']:.3f}, {q['hi']:.3f}]")
F.fig_acuity(R); plt.show()

With identical features and model, κ falls from 0.90 (synthetic) to 0.24 (real). On the synthetic data the vitals alone give κ 0.895; without them κ is 0.69. Acuity accuracy on this benchmark measures recovery of the generator.

## 5. Module 3 — Equity audit, adjusted for acuity mix
Crude undertriage rates mislead because undertriage is impossible at ESI 5. We compare observed with expected undertriage given each group's true acuity mix, test heterogeneity by permutation within acuity levels, and correct for four attributes (Holm).

In [ ]:
R["m3_equity"] = A.m3_equity(df, oof_acuity)
for col in ["language", "insurance_type", "sex", "age_group"]:
    print(f"{col:15s} Holm-adjusted p = {R['m3_equity'][col]['holm_p']}")
F.fig_equity(R); plt.show()

No attribute shows significant heterogeneity after adjustment. The audit method carries over to real data; on this synthetic data it finds no disparity.

## 6. Module 4 — The synthetic data cannot evaluate a waiting-room tool
Escalation = admitted, transferred, observation or died. The key comparison is against a model that sees **only the acuity level**.

In [ ]:
R["m4_waiting_room"], oof_syn = A.m4_waiting_room(df, hx)
m4 = R["m4_waiting_room"]
for k in ["acuity_only", "full", "delta_full_minus_acuity"]:
    c = m4["auc_low"][k]; print(f"ESI 3-5  {k:26s} {c['est']:.3f} [{c['lo']:.3f}, {c['hi']:.3f}]")
print("LWBS AUC", m4["lwbs"]["auc"])

Inside the ESI 3–5 queue the full model's AUC (0.76) equals the acuity-only model's: the synthetic disposition depends on acuity alone. Any apparent within-queue signal is just ESI 3 ranked above ESI 4 and 5. A waiting-room tool can only be judged on real outcomes.

## 7. Module 5 — Real NHAMCS validation
Folds are grouped by hospital; intervals resample hospitals; weighted results use the survey weights `PATWT`.

In [ ]:
R["m5_nhamcs"], oof_real = A.m5_nhamcs(nd)
a = R["m5_nhamcs"]["admit"]
for k, c in a["auc_low"].items():
    print(f"IMMEDR 3-5  {k:34s} {c['est']:.3f} [{c['lo']:.3f}, {c['hi']:.3f}]")
op = a["operating_points_low"]["top_20pct"]
print(f"\nFlag top 20%: recall {op['recall']:.1%} {op['recall_ci']}, lift {op['lift']}x, precision {op['precision']:.1%}")
print("calibration:", a["calibration"])
print("decision curve CI above zero for thresholds:", a["decision_curve_low"]["thresholds_with_ci_above_zero"])
print("left early AUC:", R["m5_nhamcs"]["left_early"]["auc"])

A.ART.mkdir(exist_ok=True)
(A.ART / "results.json").write_text(json.dumps(R, indent=2, default=str))
oof_syn.to_parquet(A.ART / "oof_synthetic.parquet", index=False)
pd.DataFrame(oof_real).to_parquet(A.ART / "oof_nhamcs.parquet", index=False)
F.fig_watch(R, oof_syn, pd.DataFrame(oof_real)); plt.show()
F.fig_decision_curve(R); plt.show()

Among real patients triaged as lower acuity, triage-time data ranks admission risk far better than the triage level (AUC 0.80 vs 0.64, a gain of about +0.16 with an interval far from zero; exact values printed above). Hospital-grouped validation is slightly lower than a random split (about 0.81), as expected. Survey-weighted AUC is about 0.82. After cross-fitted calibration the slope is about 0.96. Net benefit beats both "review everyone" and "review no-one" from about 2% to 35% thresholds.

*Exact figures can shift in the third decimal between library versions, because fold assignment by hospital depends on the scikit-learn version. The repository's `results.json` was produced with lightgbm 4.6.0.*

### Equity on real data

In [ ]:
F.fig_real_equity(R); plt.show()
for col, rows in R["m5_nhamcs"]["equity_low_acuity_watch"].items():
    for g, v in rows.items():
        print(f"{col:15s} {g:18s} caught {v['recall_of_watch']['est']:.0%}  observed {v['observed_rate']:.1%}  predicted {v['mean_predicted']:.1%}")

With one threshold, the share of admissions caught follows baseline risk (Medicare 79%, self-pay 26%; NH White 62%, NH Black 52%). Predicted risk is not lower than observed for any of these groups, so the gap comes from the threshold policy, not miscalibration. Choosing the threshold policy is an equity decision to make before deployment.

## 8. Test-set predictions
The competition asked for acuity predictions; here is the structured, leak-free acuity model fitted on all training data. Given Module 2, these predictions reproduce the generator and should not be read as clinical skill.

In [ ]:
test = pd.read_csv(A.find("test.csv"))
hist = pd.read_csv(A.find("patient_history.csv"))
test = test.merge(hist[["patient_id"] + hx], on="patient_id", how="left")
test["pain_score"] = test["pain_score"].replace(-1, np.nan)
test["hx_burden"] = test[hx].sum(axis=1)
model = A.clf().fit(A.make_X(df, hx), df["triage_acuity"].values)
sub = pd.DataFrame({"patient_id": test["patient_id"], "triage_acuity": model.predict(A.make_X(test, hx)).astype(int)})
sub.to_csv("submission.csv", index=False)
print(sub.shape, sub["triage_acuity"].value_counts(normalize=True).round(3).sort_index().to_dict())

## 9. Limitations
1. NHAMCS is cross-sectional: admission stands in for deterioration, and timing during the wait is unobserved.
2. Only 12 public-use variables are used; local data with full complaint text and repeat vitals should do better.
3. Hospital-grouped validation tests transfer across US hospitals, not to Nordic EDs. A silent-deployment pilot is the next step.
4. The bootstrap resamples hospitals but does not reproduce the full stratified survey design.
5. Decision support only; not a medical device.

## References
Gilboy N et al., *ESI v4*, AHRQ 2012 · Mistry B et al., *Ann Emerg Med* 2018;71:581–587 · Levin S et al., *Ann Emerg Med* 2018;71:565–574 · Hong WS et al., *PLoS One* 2018;13:e0201016 · Vickers AJ, Elkin EB, *Med Decis Making* 2006;26:565–574 · RCP, *NEWS2* 2017 · Obermeyer Z et al., *Science* 2019;366:447–453 · NCHS, *NHAMCS 2021–2022*, CDC.